# 最小 Agent vs Workflow 对比

**任务**：用户输入一个城市，系统给出 *穿衣建议*。

我们用同一个任务对比两种实现：

1. **Workflow 版本**：开发者写死调用顺序 → `get_weather` → `LLM 总结`。
2. **Agent 版本**：LLM 自主决定要不要调 `get_weather`，以及调几次。

通过这个对比，体会 *workflow 的可控性* 与 *agent 的灵活性*。

## 0. 环境准备

需要在仓库根目录的 `.env` 中设置 `ANTHROPIC_API_KEY`。

In [ ]:
import os, sys, json, random
sys.path.append(os.path.abspath('../..'))
from utils.llm_client import LLMClient

client = LLMClient()
client.provider, client.model

## 1. 模拟一个天气工具

为了离线可跑，我们用一个伪天气数据，真实场景接入和风/OpenWeather API 即可。

In [ ]:
_FAKE = {
    '北京': {'temp_c': 8, 'condition': '多云', 'wind': '北风 3 级'},
    '上海': {'temp_c': 15, 'condition': '小雨', 'wind': '东风 2 级'},
    '广州': {'temp_c': 24, 'condition': '晴', 'wind': '南风 1 级'},
}

def get_weather(city: str) -> dict:
    """模拟天气查询。返回 {temp_c, condition, wind}。"""
    return _FAKE.get(city, {'temp_c': 20, 'condition': '未知', 'wind': '未知'})

get_weather('北京')

## 2. Workflow 实现

调用顺序写死：`get_weather` → `LLM 总结`。

In [ ]:
def workflow_outfit_advice(city: str) -> str:
    weather = get_weather(city)
    prompt = (
        f'城市 {city} 当前天气：温度 {weather["temp_c"]}°C，'
        f'{weather["condition"]}，{weather["wind"]}。'
        '请给出 1-2 句穿衣建议，简洁实用。'
    )
    out = client.chat([{'role': 'user', 'content': prompt}])
    return out['text']

print(workflow_outfit_advice('北京'))

## 3. Agent 实现（Anthropic Tool Use）

把 `get_weather` 注册为工具，让 Claude 自主决定调不调、调几次。

关键：循环结构 `while resp.stop_reason == 'tool_use'`。

In [ ]:
from anthropic import Anthropic

anthropic = Anthropic()
MODEL = 'claude-sonnet-4-5'

TOOLS = [{
    'name': 'get_weather',
    'description': '查询给定城市的实时天气，返回温度、天气状况、风力。',
    'input_schema': {
        'type': 'object',
        'properties': {'city': {'type': 'string', 'description': '城市名，如 北京'}},
        'required': ['city'],
    },
}]

def run_tool(name: str, args: dict):
    if name == 'get_weather':
        return get_weather(args['city'])
    raise ValueError(f'unknown tool: {name}')

def agent_outfit_advice(user_query: str, max_steps: int = 5):
    messages = [{'role': 'user', 'content': user_query}]
    trace = []
    for step in range(max_steps):
        resp = anthropic.messages.create(
            model=MODEL,
            max_tokens=512,
            tools=TOOLS,
            messages=messages,
        )
        trace.append({'step': step, 'stop_reason': resp.stop_reason})
        if resp.stop_reason != 'tool_use':
            text = ''.join(b.text for b in resp.content if b.type == 'text')
            return text, trace
        # 否则取出 tool_use 块，本地执行，把结果回填
        messages.append({'role': 'assistant', 'content': resp.content})
        tool_results = []
        for block in resp.content:
            if block.type == 'tool_use':
                result = run_tool(block.name, block.input)
                trace.append({'tool': block.name, 'input': block.input, 'result': result})
                tool_results.append({
                    'type': 'tool_result',
                    'tool_use_id': block.id,
                    'content': json.dumps(result, ensure_ascii=False),
                })
        messages.append({'role': 'user', 'content': tool_results})
    return '[max_steps reached]', trace

answer, trace = agent_outfit_advice('我现在在北京，应该怎么穿？')
print(answer)
print('---')
for t in trace: print(t)

## 4. 对比：体现 Agent 灵活性的 query

试试 *workflow 版本无法处理* 的 query，例如：「北京和广州哪个更适合今天穿短袖？」
Agent 会自主调用两次 `get_weather`。

In [ ]:
answer, trace = agent_outfit_advice('北京和广州哪个更适合今天穿短袖？')
print(answer)
print('---')
for t in trace: print(t)

## 5. 思考

- **workflow 不能轻易扩展**：要支持「比较两个城市」必须改代码。
- **agent 灵活但贵**：每次都要让 LLM 决策，token 开销和时延都更高。
- **生产建议**：先 workflow 跑通主线，再为「开放问题」上 agent；或者用 Routing workflow 把开放问题路由到 agent，常见问题路由到 workflow。

## 进阶练习

1. 给 `get_weather` 加一个 `lang` 参数，让 agent 自己决定语言。
2. 加一个 `get_air_quality` 工具，看 agent 会不会主动调用。
3. 用 `client.usage` 比较 workflow 和 agent 的总 token 消耗。